Damped Newton method 
chi=30, truncating to 14,14 per leg
Start Newton method after 4 RG steps from T=T_c.
Damping with newton_step=0.5 is activated a couple of times (e.g. for i=3)
Here I ran up to i=9, reaching fp_error =2.3e-5. 

Here 20 eigenvalues are used (for no particular reason, I guess 10 eigenvalues would work as well). 

gilt_eps=2e-5

Compared to the previous version of the code, discrete gauge fixing when computing the Jacobian is carried out using the gauge-fixing element set of R(A), not of A. This eliminates a lot of warnings.

In [1]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [3]:
gilt_eps = 2e-5
chi = 30
trunc_shape = [14 14; 14 14; 14 14; 14 14]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true
)
Jratio = 1.0

relT=1.0
rg_steps = 10
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.03361407516010624 and became 0.0. Index CartesianIndex(1, 16, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -0.012345916839578114 and became -1.8524388175538877e-9. Index CartesianIndex(15, 2, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466


1 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.004606248308129038 and became 0.0. Index CartesianIndex(17, 3, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.0022689612252432567 and became 0.0. Index CartesianIndex(15, 2, 19, 2) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.0004254458368950628 and became 0.0. Index CartesianIndex(15, 21, 3, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -9.030177255728281e-5 and became 0.0. Index CartesianIndex(15, 1, 9, 16) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 6.0392199719057

[1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][0 1; 0 1; 0 1; 0 1]
3 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
4 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
5 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
6 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
7 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
8 [14 16; 14 16; 14 16; 14 16][0 1; 0 1; 0 1; 0 1]
9 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
10 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
11 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]


In [5]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = truncate_blocks(traj[4], trunc_shape)
for i in 1:30
    println("i=",i)

    RA = gilt_with_cont_gauge(A[i], gilt_pars; trunc_shape = trunc_shape);
    RA, accepted_elements[i] = fix_discrete_gauge(RA; tol = 1e-7);
    
    A[i], _ = fix_discrete_gauge(ju_to_py(A[i]), accepted_elements[i])
    A[i] = py_to_ju(A[i])

    e0 = embedded_distance(RA, A[i])
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RA.shape)
    flush(stdout)
    
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 20, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 1.0
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]

        RAnew = gilt_with_cont_gauge(Anew, gilt_pars; trunc_shape = trunc_shape);
        RAnew, accepted_elements_new = fix_discrete_gauge(RAnew; tol = 1e-7);
    
        Anew, _ = fix_discrete_gauge(ju_to_py(Anew), accepted_elements_new)
        Anew = py_to_ju(Anew)

        enew = embedded_distance(RAnew, Anew)
        
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnew.shape)
        if enew < 0.8 * e0 && Anew.shape == RAnew.shape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=1
||R(A[i])-A[i]||= 0.0578969047685301
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
Dict{Any, Any}((1, "N") => 28, (1, "W") => 22, (1, "S") => 25, (1, "E") => 22, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  25 eigenvalues converged
│ *  norm of residuals = (7.164096861493545e-65, 1.599769124136846e-51, 1.1908978292914387e-49, 1.564860558549025e-38, 8.74945716115196e-38, 1.2692152742413536e-33, 1.2692152742413536e-33, 8.46082263179831e-27, 2.983221069705698e-26, 2.983221069705698e-26, 9.419330068723644e-25, 9.419330068723644e-25, 8.70108221405038e-25, 8.70108221405038e-25, 1.4032283428104842e-20, 6.8761681659638535e-22, 6.095610509561633e-18, 1.8054755546750694e-16, 4.657553732646705e-15, 5.633651790500182e-17, 1.2331863095089688e-16, 1.2331863095089688e-16, 2.203398713841081e-16, 2.203398713841081e-16, 3.4610310919329045e-16)
└ *  number of operations = 63


EIGENVALUES (INITIAL):
1.9851306569862872 + 0.0im
-0.9350311749671247 + 0.0im
-0.9250409357582516 + 0.0im
0.5575703150261458 + 0.0im
0.5508470460463115 + 0.0im
0.0008033037587934081 + 0.41917115765206064im
0.0008033037587934081 - 0.41917115765206064im
-0.31042499672460916 + 0.0im
0.00027089770706557035 + 0.3094635220071427im
0.00027089770706557035 - 0.3094635220071427im
0.07537126113133812 + 0.2745166168592897im
0.07537126113133812 - 0.2745166168592897im
-0.07545860525987161 + 0.2736474550063157im
-0.07545860525987161 - 0.2736474550063157im
0.2659620236831136 + 0.0im
-0.24652354993478284 + 0.0im
0.2429820686201013 + 0.0im
0.22917368489520884 + 0.0im
0.2101327063554566 + 0.0im
-0.19347801107243495 + 0.0im
-0.018466944381779173 + 0.18687708991215146im
-0.018466944381779173 - 0.18687708991215146im
0.019517292513691185 + 0.18629811918536732im
0.019517292513691185 - 0.18629811918536732im
-0.18384958017308098 + 0.0im
||deltaA[i]||= 0.1369329913656389
newton_step= 1.0
fp_error= 0.027876401300

┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  20 eigenvalues converged
│ *  norm of residuals = (2.3627210825396407e-60, 1.60767280005667e-45, 1.6750325215882488e-45, 4.6186299944140286e-38, 1.76692928880803e-35, 2.2284521982849118e-33, 2.2284521982849118e-33, 3.104161507457014e-28, 7.384214881983243e-22, 7.384214881983243e-22, 2.0433697177633956e-21, 2.0433697177633956e-21, 1.054912070776746e-17, 1.054912070776746e-17, 1.39485164383448e-20, 1.39485164383448e-20, 3.877311986922718e-19, 3.877311986922718e-19, 1.4350211761152876e-15, 2.1232624877696176e-16)
└ *  number of operations = 61


EIGENVALUES (INITIAL):
1.9885336234545468 + 0.0im
-0.9728411821028574 + 0.0im
-0.962002231189387 + 0.0im
0.6551718749455927 + 0.0im
0.6001368254821599 + 0.0im
-0.0058657672334424295 + 0.5273635389054718im
-0.0058657672334424295 - 0.5273635389054718im
-0.40745239946011397 + 0.0im
0.006916074004776787 + 0.3087899432683377im
0.006916074004776787 - 0.3087899432683377im
-0.12273059855579216 + 0.26033905440092014im
-0.12273059855579216 - 0.26033905440092014im
0.28415220882937064 + 0.008974325890998579im
0.28415220882937064 - 0.008974325890998579im
0.12068778368332606 + 0.2515275029168003im
0.12068778368332606 - 0.2515275029168003im
0.0010522094385849484 + 0.26821035621408135im
0.0010522094385849484 - 0.26821035621408135im
0.2625985908530338 + 0.0im
-0.2329412580188482 + 0.0im
||deltaA[i]||= 0.0725171806914249
newton_step= 1.0
fp_error= 0.023049726902213838
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
newton_step= 0.5
fp_error= 0.011983014562540606
shapes:[14 14; 14 14; 14 

...

┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  24 eigenvalues converged
│ *  norm of residuals = (1.342650926020431e-62, 5.044079064393194e-48, 2.2497473722993315e-48, 1.3681652718147686e-41, 9.025866743398885e-37, 4.517704249982164e-34, 4.517704249982164e-34, 8.8702865610842e-25, 1.668513991682904e-25, 1.0580949188409832e-22, 1.0580949188409832e-22, 7.764604004416954e-19, 3.169562441955833e-21, 3.169562441955833e-21, 9.40769815147694e-21, 9.40769815147694e-21, 5.182015721995871e-20, 5.182015721995871e-20, 2.5949248287533183e-17, 2.2157662418822105e-19, 2.4326995584151233e-17, 2.4326995584151233e-17, 1.0488950099097171e-13, 8.293642272742239e-17)
└ *  number of operations = 62


EIGENVALUES (INITIAL):
1.9984737592536626 + 0.0im
-0.9949779309927802 + 0.0im
-0.9867459857838191 + 0.0im
0.7045226425700934 + 0.0im
0.5951575031587818 + 0.0im
-0.01572403379013362 + 0.5009073653303706im
-0.01572403379013362 - 0.5009073653303706im
0.3495837333848587 + 0.0im
-0.3373488766677214 + 0.0im
-0.14890977349423973 + 0.24309533518690785im
-0.14890977349423973 - 0.24309533518690785im
0.28437018957373095 + 0.0im
0.14882940463099986 + 0.23321337296168632im
0.14882940463099986 - 0.23321337296168632im
-0.019982580213115317 + 0.27381709250515623im
-0.019982580213115317 - 0.27381709250515623im
0.010534142158243685 + 0.26705978923315005im
0.010534142158243685 - 0.26705978923315005im
0.2607924786399688 + 0.0im
-0.25011969770648274 + 0.0im
0.003957428137527322 + 0.23195205424630336im
0.003957428137527322 - 0.23195205424630336im
0.2235344591101607 + 0.0im
-0.21773998358158114 + 0.0im
||deltaA[i]||= 0.0005674253713641227
newton_step= 1.0
fp_error= 4.580465006536098e-5
shapes:[14 14; 14 14; 

┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  24 eigenvalues converged
│ *  norm of residuals = (2.60123803695123e-63, 1.0374107832817883e-49, 2.44427747816923e-48, 3.9067814235356106e-38, 3.9067814235356106e-38, 2.035823365757872e-35, 2.035823365757872e-35, 1.6620338561523823e-28, 1.4144294548270815e-27, 2.1784339763025762e-23, 2.1784339763025762e-23, 2.271919738220294e-20, 2.5971604300769885e-22, 2.5971604300769885e-22, 1.8326953973070198e-21, 1.8326953973070198e-21, 1.6535007707170508e-20, 1.6535007707170508e-20, 4.277090532298772e-18, 9.583294456750111e-20, 1.349784812125646e-17, 1.349784812125646e-17, 1.8522628386386495e-14, 1.2874995392036593e-16)
└ *  number of operations = 63


EIGENVALUES (INITIAL):
1.9983104937966267 + 0.0im
-0.9943532076902883 + 0.0im
-0.9866326275066928 + 0.0im
0.6002275206584525 + 0.010690768688341259im
0.6002275206584525 - 0.010690768688341259im
-0.007489182838621315 + 0.5038980556524931im
-0.007489182838621315 - 0.5038980556524931im
0.39148182836231993 + 0.0im
-0.35132816931447347 + 0.0im
-0.1483031612549366 + 0.24673095037648082im
-0.1483031612549366 - 0.24673095037648082im
0.2841126054779972 + 0.0im
0.15182116128995465 + 0.2336186499297582im
0.15182116128995465 - 0.2336186499297582im
-0.02012914855744167 + 0.2733899574210687im
-0.02012914855744167 - 0.2733899574210687im
0.009992196048988794 + 0.26329867147075575im
0.009992196048988794 - 0.26329867147075575im
0.25947718810744724 + 0.0im
-0.24514091333275875 + 0.0im
0.006266498707928946 + 0.235546446640072im
0.006266498707928946 - 0.235546446640072im
0.2235199409739687 + 0.0im
-0.21814146814493274 + 0.0im
||deltaA[i]||= 0.00013356844467841226
newton_step= 1.0
fp_error= 2.30725855554108

LoadError: InterruptException: